In [1]:

import sys
import subprocess
import importlib.util

for module, package in {
    "gradio": "gradio>=5,<7",
    "plotly": "plotly>=5,<7",
    "pandas": "pandas>=2",
    "numpy": "numpy>=1.24"
}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import io
import uuid
import zipfile
import urllib.request
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr

BASE_DIR = Path("/content/academic_hci")
BASE_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
ZIP_PATH = BASE_DIR / "student_performance.zip"
USABILITY_PATH = BASE_DIR / "usability_results.csv"

STUDY_LABELS = {
    1: "Kurang dari 2 jam",
    2: "2–5 jam",
    3: "5–10 jam",
    4: "Lebih dari 10 jam"
}

SCHOOL_LABELS = {
    "GP": "Gabriel Pereira",
    "MS": "Mousinho da Silveira"
}

SEX_LABELS = {
    "F": "Perempuan",
    "M": "Laki-laki"
}

COLORS = {
    "Lulus": "#2563EB",
    "Tidak Lulus": "#EA580C"
}


def download_dataset():
    if ZIP_PATH.exists():
        try:
            with zipfile.ZipFile(ZIP_PATH) as archive:
                if archive.testzip() is None:
                    return ZIP_PATH
        except (zipfile.BadZipFile, OSError):
            pass

    request = urllib.request.Request(
        DATA_URL,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(request, timeout=120) as response:
        content = response.read()

    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError("Arsip dataset tidak valid.")

    ZIP_PATH.write_bytes(content)
    return ZIP_PATH


def find_csv(blob, target="student-mat.csv"):
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        for name in archive.namelist():
            if name.lower().endswith(target.lower()):
                return archive.read(name)

        for name in archive.namelist():
            if name.lower().endswith(".zip"):
                try:
                    result = find_csv(archive.read(name), target)
                    if result is not None:
                        return result
                except (zipfile.BadZipFile, OSError):
                    continue

    return None


def load_dataset():
    path = download_dataset()
    csv_content = find_csv(path.read_bytes())

    if csv_content is None:
        raise FileNotFoundError("Dataset student-mat.csv tidak ditemukan.")

    data = pd.read_csv(io.BytesIO(csv_content), sep=";")

    required = [
        "school", "sex", "age", "studytime",
        "absences", "G1", "G2", "G3",
        "failures", "internet"
    ]

    if not set(required).issubset(data.columns):
        raise ValueError("Struktur dataset tidak sesuai.")

    numeric = [
        "age", "studytime", "absences",
        "G1", "G2", "G3", "failures"
    ]

    for column in numeric:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    data = data.dropna(subset=required).copy()

    for column in ["G1", "G2", "G3"]:
        data = data[data[column].between(0, 20)]

    data = data[data["studytime"].isin(STUDY_LABELS)]

    if data.empty:
        raise ValueError("Tidak ada data valid.")

    data["Sekolah"] = data["school"].map(SCHOOL_LABELS)
    data["Jenis Kelamin"] = data["sex"].map(SEX_LABELS)
    data["Waktu Belajar"] = data["studytime"].map(STUDY_LABELS)
    data["Akses Internet"] = data["internet"].map({
        "yes": "Ya",
        "no": "Tidak"
    })
    data["Status"] = np.where(data["G3"] >= 10, "Lulus", "Tidak Lulus")

    return data.reset_index(drop=True)


df = load_dataset()

MIN_AGE = int(df["age"].min())
MAX_AGE = int(df["age"].max())
SCHOOLS = sorted(df["Sekolah"].dropna().unique().tolist())


def filter_data(school, sex, study, internet, min_age, max_age):
    if min_age is None or max_age is None:
        raise gr.Error("Pilih rentang usia.")

    if int(min_age) > int(max_age):
        raise gr.Error("Usia minimum melebihi usia maksimum.")

    filtered = df.copy()

    filters = {
        "Sekolah": school,
        "Jenis Kelamin": sex,
        "Waktu Belajar": study,
        "Akses Internet": internet
    }

    for column, value in filters.items():
        if value != "Semua":
            filtered = filtered[filtered[column] == value]

    return filtered[
        filtered["age"].between(int(min_age), int(max_age))
    ].copy()


def create_dashboard(school, sex, study, internet, min_age, max_age):
    data = filter_data(school, sex, study, internet, min_age, max_age)

    if data.empty:
        raise gr.Error("Tidak ada data untuk filter tersebut.")

    total = len(data)
    mean_grade = data["G3"].mean()
    median_grade = data["G3"].median()
    pass_rate = data["G3"].ge(10).mean() * 100
    mean_absence = data["absences"].mean()

    summary = f"""
## Ringkasan Kinerja Akademik

| Indikator | Nilai |
|---|---:|
| Jumlah siswa | {total} |
| Rata-rata nilai akhir | {mean_grade:.2f} |
| Median nilai akhir | {median_grade:.2f} |
| Tingkat kelulusan | {pass_rate:.2f}% |
| Rata-rata ketidakhadiran | {mean_absence:.2f} |
| Persentase data terpilih | {total / len(df) * 100:.2f}% |

Nilai akhir menggunakan skala 0–20. Status lulus ditentukan berdasarkan G3 ≥ 10.
"""

    fig_distribution = px.histogram(
        data,
        x="G3",
        color="Status",
        nbins=21,
        title="Distribusi Nilai Akhir",
        color_discrete_map=COLORS,
        labels={"G3": "Nilai Akhir"}
    )

    fig_distribution.update_layout(
        template="plotly_white",
        height=420,
        yaxis_title="Jumlah Siswa"
    )

    study_stats = (
        data.groupby(["studytime", "Waktu Belajar"])
        .agg(
            Nilai_Rata_Rata=("G3", "mean"),
            Jumlah_Siswa=("G3", "size")
        )
        .reset_index()
        .sort_values("studytime")
    )

    fig_study = px.bar(
        study_stats,
        x="Waktu Belajar",
        y="Nilai_Rata_Rata",
        text="Nilai_Rata_Rata",
        hover_data=["Jumlah_Siswa"],
        title="Nilai Berdasarkan Waktu Belajar"
    )

    fig_study.update_traces(
        marker_color="#2563EB",
        texttemplate="%{text:.2f}",
        textposition="outside"
    )

    fig_study.update_layout(
        template="plotly_white",
        height=420,
        yaxis_range=[0, 22]
    )

    fig_absence = px.scatter(
        data,
        x="absences",
        y="G3",
        color="Status",
        color_discrete_map=COLORS,
        hover_data=["Sekolah", "Waktu Belajar", "G1", "G2"],
        title="Ketidakhadiran dan Nilai Akhir",
        labels={
            "absences": "Jumlah Ketidakhadiran",
            "G3": "Nilai Akhir"
        },
        opacity=0.7
    )

    fig_absence.update_layout(
        template="plotly_white",
        height=420
    )

    school_stats = (
        data.groupby("Sekolah")
        .agg(
            Rata_Rata=("G3", "mean"),
            Jumlah=("G3", "size")
        )
        .reset_index()
    )

    fig_school = px.bar(
        school_stats,
        x="Sekolah",
        y="Rata_Rata",
        text="Rata_Rata",
        hover_data=["Jumlah"],
        title="Rata-rata Nilai per Sekolah"
    )

    fig_school.update_traces(
        marker_color="#0D9488",
        texttemplate="%{text:.2f}",
        textposition="outside"
    )

    fig_school.update_layout(
        template="plotly_white",
        height=420,
        yaxis_range=[0, 22]
    )

    correlation = data[
        ["age", "studytime", "absences", "failures", "G1", "G2", "G3"]
    ].corr(numeric_only=True)

    fig_corr = px.imshow(
        correlation,
        zmin=-1,
        zmax=1,
        color_continuous_scale="RdBu_r",
        title="Korelasi Variabel Akademik",
        aspect="auto"
    )

    fig_corr.update_layout(
        template="plotly_white",
        height=480
    )

    status_stats = (
        data["Status"]
        .value_counts()
        .reindex(["Lulus", "Tidak Lulus"], fill_value=0)
        .reset_index()
    )

    status_stats.columns = ["Status", "Jumlah"]

    fig_status = px.bar(
        status_stats,
        x="Status",
        y="Jumlah",
        color="Status",
        text="Jumlah",
        color_discrete_map=COLORS,
        title="Distribusi Status Akademik"
    )

    fig_status.update_layout(
        template="plotly_white",
        height=420,
        showlegend=False
    )

    details = data[
        [
            "Sekolah", "Jenis Kelamin", "age",
            "Waktu Belajar", "Akses Internet",
            "absences", "G1", "G2", "G3", "Status"
        ]
    ].reset_index(drop=True)

    def safe_corr(column):
        if (
            len(data) < 3
            or data[column].nunique() < 2
            or data["G3"].nunique() < 2
        ):
            return "Tidak tersedia"

        value = data[[column, "G3"]].corr().iloc[0, 1]
        return f"{value:.3f}" if pd.notna(value) else "Tidak tersedia"

    insight = f"""
## Insight Akademik

**Jumlah siswa:** {total}

**Rata-rata nilai akhir:** {mean_grade:.2f}

**Tingkat kelulusan:** {pass_rate:.2f}%

**Korelasi waktu belajar dengan nilai:** {safe_corr("studytime")}

**Korelasi ketidakhadiran dengan nilai:** {safe_corr("absences")}

Korelasi merupakan hubungan statistik dan tidak membuktikan sebab-akibat.
"""

    export_path = BASE_DIR / f"academic_{uuid.uuid4().hex[:12]}.csv"
    details.to_csv(export_path, index=False, encoding="utf-8-sig")

    return (
        summary,
        fig_distribution,
        fig_study,
        fig_absence,
        fig_school,
        fig_corr,
        fig_status,
        details,
        insight,
        str(export_path)
    )


TASKS = [
    "T1 - Menemukan rata-rata nilai akhir",
    "T2 - Membandingkan waktu belajar",
    "T3 - Menemukan persentase kelulusan",
    "T4 - Membandingkan nilai antar sekolah",
    "T5 - Mengunduh data CSV"
]


def save_usability(participant, task, success, duration, difficulty):
    participant = str(participant or "").strip()

    if not participant:
        raise gr.Error("Kode partisipan wajib diisi.")

    if not task or not success or difficulty is None:
        raise gr.Error("Lengkapi seluruh pilihan evaluasi.")

    if duration is None or not np.isfinite(float(duration)) or float(duration) <= 0:
        raise gr.Error("Durasi harus lebih besar dari nol.")

    record = pd.DataFrame([{
        "Partisipan": participant,
        "Tugas": task,
        "Berhasil": int(success == "Berhasil"),
        "Durasi_Detik": float(duration),
        "Kemudahan_1_5": int(difficulty),
        "Waktu": datetime.now().isoformat(timespec="seconds")
    }])

    if USABILITY_PATH.exists():
        previous = pd.read_csv(
            USABILITY_PATH,
            dtype={"Partisipan": str}
        )
        records = pd.concat([previous, record], ignore_index=True)
    else:
        records = record

    records.to_csv(
        USABILITY_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    success_rate = records["Berhasil"].mean() * 100
    mean_time = records["Durasi_Detik"].mean()
    mean_ease = records["Kemudahan_1_5"].mean()

    report = f"""
## Hasil Evaluasi Usability

| Indikator | Nilai |
|---|---:|
| Jumlah percobaan | {len(records)} |
| Partisipan unik | {records["Partisipan"].nunique()} |
| Task Success Rate | {success_rate:.2f}% |
| Average Task Time | {mean_time:.2f} detik |
| Rata-rata kemudahan | {mean_ease:.2f}/5 |
"""

    return report, records, str(USABILITY_PATH)


with gr.Blocks(title="Academic Performance Dashboard") as demo:

    gr.Markdown("""
# Dashboard Interaktif Kinerja Akademik

### Human-Computer Interaction & Data Visualization

Analisis data akademik menggunakan visualisasi interaktif berbasis dataset UCI Student Performance.
""")

    with gr.Row():
        school_input = gr.Dropdown(
            ["Semua"] + SCHOOLS,
            value="Semua",
            label="Sekolah"
        )

        sex_input = gr.Dropdown(
            ["Semua", "Perempuan", "Laki-laki"],
            value="Semua",
            label="Jenis Kelamin"
        )

        study_input = gr.Dropdown(
            ["Semua"] + list(STUDY_LABELS.values()),
            value="Semua",
            label="Waktu Belajar"
        )

    with gr.Row():
        internet_input = gr.Dropdown(
            ["Semua", "Ya", "Tidak"],
            value="Semua",
            label="Akses Internet"
        )

        min_age_input = gr.Slider(
            minimum=MIN_AGE,
            maximum=MAX_AGE,
            value=MIN_AGE,
            step=1,
            label="Usia Minimum"
        )

        max_age_input = gr.Slider(
            minimum=MIN_AGE,
            maximum=MAX_AGE,
            value=MAX_AGE,
            step=1,
            label="Usia Maksimum"
        )

    analyze_button = gr.Button(
        "Tampilkan Analisis",
        variant="primary"
    )

    with gr.Tab("Ringkasan"):
        summary_output = gr.Markdown()

    with gr.Tab("Distribusi Nilai"):
        distribution_output = gr.Plot()

    with gr.Tab("Waktu Belajar"):
        study_output = gr.Plot()

    with gr.Tab("Ketidakhadiran"):
        absence_output = gr.Plot()

    with gr.Tab("Perbandingan Sekolah"):
        school_output = gr.Plot()

    with gr.Tab("Korelasi"):
        correlation_output = gr.Plot()

    with gr.Tab("Status Akademik"):
        status_output = gr.Plot()

    with gr.Tab("Detail Data"):
        detail_output = gr.Dataframe(interactive=False)

    with gr.Tab("Insight"):
        insight_output = gr.Markdown()

    with gr.Tab("Unduh CSV"):
        export_output = gr.File(label="Hasil Analisis CSV")

    dashboard_inputs = [
        school_input,
        sex_input,
        study_input,
        internet_input,
        min_age_input,
        max_age_input
    ]

    dashboard_outputs = [
        summary_output,
        distribution_output,
        study_output,
        absence_output,
        school_output,
        correlation_output,
        status_output,
        detail_output,
        insight_output,
        export_output
    ]

    analyze_button.click(
        fn=create_dashboard,
        inputs=dashboard_inputs,
        outputs=dashboard_outputs
    )

    demo.load(
        fn=create_dashboard,
        inputs=dashboard_inputs,
        outputs=dashboard_outputs
    )

    gr.Markdown("## Evaluasi Usability")

    with gr.Accordion("Form Evaluasi", open=False):
        participant_input = gr.Textbox(
            label="Kode Partisipan",
            placeholder="P01"
        )

        task_input = gr.Dropdown(
            choices=TASKS,
            label="Tugas Pengujian"
        )

        success_input = gr.Radio(
            ["Berhasil", "Gagal"],
            label="Hasil Tugas"
        )

        duration_input = gr.Number(
            label="Durasi (detik)",
            minimum=0
        )

        difficulty_input = gr.Radio(
            [1, 2, 3, 4, 5],
            label="Kemudahan (1–5)"
        )

        save_button = gr.Button(
            "Simpan Evaluasi",
            variant="primary"
        )

        usability_report = gr.Markdown()
        usability_table = gr.Dataframe(interactive=False)
        usability_download = gr.File(label="Unduh Hasil Evaluasi")

        save_button.click(
            fn=save_usability,
            inputs=[
                participant_input,
                task_input,
                success_input,
                duration_input,
                difficulty_input
            ],
            outputs=[
                usability_report,
                usability_table,
                usability_download
            ]
        )

    gr.Markdown("""
**Sumber:** [UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance)

Data merupakan data historis siswa sekolah menengah Portugal. Analisis bersifat deskriptif dan tidak membuktikan hubungan sebab-akibat.
""")

demo.launch(share=True, show_error=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d30e684f9280c4c79.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
